In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Warehouse'))

In [ ]:
from Warehouse_Builder import Warehouse_Builder, WarehouseConfig, AisleConfig

## Parameters
Edit the config below to define the warehouse layout.

- `total_aisles` — total number of aisles to generate
- `aisle_splits` — proportion of total aisles per type (must sum to 1.0)
- `storage_sizes` — one size for uniform aisles, multiple + `size_probabilities` for distributed aisles

In [ ]:
config = WarehouseConfig(
    total_aisles=10,
    aisle_splits=[0.5, 0.3, 0.2],
    aisle_configs=[
        AisleConfig(
            handling_type='conveyable',
            storage_type='food',
            unit_type='pallet',
            bayXPerAisle=4,
            bayYPerAisle=6,
            storage_sizes=['medium'],
        ),
        AisleConfig(
            handling_type='non-conveyable',
            storage_type='furniture',
            unit_type='pallet',
            bayXPerAisle=4,
            bayYPerAisle=8,
            storage_sizes=['small', 'large'],
            size_probabilities=[0.6, 0.4],
        ),
        AisleConfig(
            handling_type='conveyable',
            storage_type='electronic',
            unit_type='singleton',
            bayXPerAisle=3,
            bayYPerAisle=4,
            storage_sizes=['extra_large'],
        ),
    ]
)

## Build

In [ ]:
warehouse = Warehouse_Builder().from_config(config).build()
print(f'Aisles : {len(warehouse.aisles)}')
print(f'Total bins: {len(warehouse.bins)}')

## Aisle Summary

In [ ]:
import pandas as pd

rows = []
for aisle in warehouse.aisles:
    size_counts = {}
    for b in aisle.bins:
        size_counts[b.storage_size] = size_counts.get(b.storage_size, 0) + 1
    rows.append({
        'aisle_id'             : aisle.aisle_id,
        'handling_type'        : aisle.handling_type,
        'storage_type'         : aisle.storage_type,
        'unit_type'            : aisle.unit_type,
        'storage_handling_type': aisle.storage_handling_type,
        'bays (X x Y)'         : f'{aisle.bayXPerAisle} x {aisle.bayYPerAisle}',
        'total_bins'           : len(aisle.bins),
        'size_counts'          : size_counts,
    })

pd.DataFrame(rows)

## Bin Grid per Aisle
Rows = Y levels, columns = X positions. Cell value is the storage size of that bin.

In [ ]:
aisle_id = 1   # <-- change to inspect a different aisle

aisle = next(a for a in warehouse.aisles if a.aisle_id == aisle_id)
grid = {}
for b in aisle.bins:
    grid[(b.bayY, b.bayX)] = b.storage_size

index   = range(aisle.bayYPerAisle, 0, -1)
columns = range(1, aisle.bayXPerAisle + 1)
df = pd.DataFrame(
    [[grid[(y, x)] for x in columns] for y in index],
    index  =[f'Y={y}' for y in index],
    columns=[f'X={x}' for x in columns],
)
print(f'Aisle {aisle_id}  |  handling: {aisle.handling_type}  |  storage: {aisle.storage_type}  |  combined: {aisle.storage_handling_type}')
df

## Warehouse-Wide Size Distribution

In [ ]:
import matplotlib.pyplot as plt

size_counts = {}
for b in warehouse.bins:
    size_counts[b.storage_size] = size_counts.get(b.storage_size, 0) + 1

sizes  = list(size_counts.keys())
counts = list(size_counts.values())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sizes, counts)
ax.set_xlabel('Storage size')
ax.set_ylabel('Bin count')
ax.set_title('Bin count by storage size')
for i, v in enumerate(counts):
    ax.text(i, v + 0.3, str(v), ha='center')
plt.tight_layout()
plt.show()